# 🔗 GNN Bar-to-Baseline Grouping Training

This notebook trains a Graph Neural Network to learn which bars belong to which baselines in bar charts.

**Training data**: Generated synthetically using `generator.py` - no manual annotation required!

## Setup

In [ ]:
# Install dependencies
!pip install torch-geometric torch-scatter torch-sparse -q
!pip install tqdm matplotlib numpy -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
import json
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from google.colab import files
import zipfile
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Upload Training Data

Upload the `training_data.zip` generated by `package_training_data.py`

In [ ]:
# Upload the training data ZIP
uploaded = files.upload()

# Extract
zip_filename = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_filename, 'r') as zf:
    zf.extractall('training_data')
    
print("Extracted files:")
!ls -la training_data/

In [ ]:
# Load GNN training data
with open('training_data/gnn_training_data.json', 'r') as f:
    gnn_data = json.load(f)
    
print(f"Loaded {len(gnn_data)} training samples")
print(f"Sample keys: {list(gnn_data[0].keys()) if gnn_data else 'empty'}")

## Data Loading

In [ ]:
class GNNDataset:
    """Convert JSON training data to PyTorch Geometric Data objects."""
    
    def __init__(self, json_data, normalize=True):
        self.samples = json_data
        self.normalize = normalize
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        return self.build_graph(sample)
    
    def build_graph(self, sample):
        """Build PyG Data object from sample."""
        bars = sample['bars']
        baselines = sample['baselines']
        edges = sample['edges']
        
        # Node features: [cx, cy, w, h, is_bar, is_baseline, conf, 0]
        node_features = []
        
        # Bar nodes (type 0)
        for bar in bars:
            x0, y0, x1, y1 = bar['xyxy']
            cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
            w, h = x1 - x0, y1 - y0
            node_features.append([cx, cy, w, h, 1.0, 0.0, bar.get('conf', 1.0), 0.0])
        
        # Baseline nodes (type 1)
        for bl in baselines:
            cx = (bl['x_range'][0] + bl['x_range'][1]) / 2
            cy = bl['y_pixel']
            w = bl['x_range'][1] - bl['x_range'][0]
            h = 5.0  # Baselines are thin lines
            node_features.append([cx, cy, w, h, 0.0, 1.0, 1.0, 1.0])
        
        # Normalize features
        node_features = np.array(node_features, dtype=np.float32)
        if self.normalize and len(node_features) > 0:
            node_features[:, :4] = node_features[:, :4] / 800.0  # Assume ~800px images
        
        # Edge index and labels
        edge_index = []
        edge_labels = []
        
        num_bars = len(bars)
        for edge in edges:
            src = edge['bar_idx']  # Bar node
            dst = num_bars + edge['baseline_idx']  # Baseline node
            edge_index.append([src, dst])
            edge_index.append([dst, src])  # Bidirectional
            edge_labels.append(float(edge['is_connected']))
            edge_labels.append(float(edge['is_connected']))
        
        if len(edge_index) == 0:
            edge_index = [[0, 0]]  # Dummy edge
            edge_labels = [0.0]
        
        return Data(
            x=torch.tensor(node_features, dtype=torch.float),
            edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
            y=torch.tensor(edge_labels, dtype=torch.float),
            num_bars=num_bars,
            num_baselines=len(baselines)
        )

In [ ]:
# Create dataset and split
dataset = GNNDataset(gnn_data)

# Split: 80% train, 10% val, 10% test
n = len(dataset)
train_size = int(0.8 * n)
val_size = int(0.1 * n)
test_size = n - train_size - val_size

indices = np.random.permutation(n)
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

train_data = [dataset[i] for i in train_indices]
val_data = [dataset[i] for i in val_indices]
test_data = [dataset[i] for i in test_indices]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

In [ ]:
# Create data loaders
from torch_geometric.loader import DataLoader as PyGDataLoader

train_loader = PyGDataLoader(train_data, batch_size=32, shuffle=True)
val_loader = PyGDataLoader(val_data, batch_size=32)
test_loader = PyGDataLoader(test_data, batch_size=32)

## GNN Model

In [ ]:
class BarBaselineGNN(nn.Module):
    """
    GNN for bar-to-baseline grouping.
    
    Architecture:
    - 2-layer GCN for node embeddings
    - Edge classifier predicts bar-baseline connections
    """
    
    def __init__(self, node_features=8, hidden_dim=64, num_layers=2):
        super().__init__()
        
        # Node embedding layers
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(node_features, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        
        # Edge classifier
        self.edge_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        
        # Node embeddings
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=0.2, training=self.training)
        
        # Edge predictions
        src, dst = edge_index
        edge_features = torch.cat([x[src], x[dst]], dim=1)
        edge_logits = self.edge_classifier(edge_features).squeeze(-1)
        
        return edge_logits

model = BarBaselineGNN(node_features=8, hidden_dim=64, num_layers=2).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

# Weighted BCE loss (positive edges are rare)
pos_weight = torch.tensor([5.0]).to(device)  # Weight positive examples more

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        logits = model(batch)
        loss = F.binary_cross_entropy_with_logits(logits, batch.y, pos_weight=pos_weight)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == batch.y).sum().item()
        total += len(batch.y)
    
    return total_loss / len(loader.dataset), correct / total


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    # Track precision/recall
    tp, fp, fn = 0, 0, 0
    
    for batch in loader:
        batch = batch.to(device)
        
        logits = model(batch)
        loss = F.binary_cross_entropy_with_logits(logits, batch.y, pos_weight=pos_weight)
        
        total_loss += loss.item() * batch.num_graphs
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == batch.y).sum().item()
        total += len(batch.y)
        
        # Precision/Recall
        tp += ((preds == 1) & (batch.y == 1)).sum().item()
        fp += ((preds == 1) & (batch.y == 0)).sum().item()
        fn += ((preds == 0) & (batch.y == 1)).sum().item()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return total_loss / len(loader.dataset), correct / total, f1

In [ ]:
# Training loop
NUM_EPOCHS = 50
best_val_f1 = 0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_loss, val_acc, val_f1 = eval_epoch(model, val_loader)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_gnn_model.pt')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        print(f"  Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

print(f"\nBest Val F1: {best_val_f1:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses, label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_title('Loss Curves')

axes[1].plot(train_accs, label='Train')
axes[1].plot(val_accs, label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_title('Accuracy Curves')

plt.tight_layout()
plt.show()

## Evaluation

In [ ]:
# Load best model and evaluate on test set
model.load_state_dict(torch.load('best_gnn_model.pt'))
test_loss, test_acc, test_f1 = eval_epoch(model, test_loader)

print(f"Test Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1 Score: {test_f1:.4f}")

## Export Model

In [ ]:
# Save model for deployment
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'node_features': 8,
        'hidden_dim': 64,
        'num_layers': 2
    },
    'test_f1': test_f1,
    'test_accuracy': test_acc
}, 'bar_baseline_gnn_final.pt')

# Download
files.download('bar_baseline_gnn_final.pt')
print("Model exported and downloaded!")